### `nz_plots_magabs.ipynb`
---------------

How much does absolute error sigma_alpha = 0.1 widen the final error bars?

In [ ]:
import numpy as np
import pandas as pd
import importlib
import json
import matplotlib.pyplot as plt

from pathlib import Path

import src.statistics.spline as spline
import src.analysis.plots as plots
import src.statistics.corrfiles as cf
import src.statistics.systematics as sy

importlib.reload(spline)
importlib.reload(sy)

ROOT = cf.get_base_dir()

# systematics figures live in their own directory so they cannot clobber paper figures
FIGURES_ROOT = ROOT / "paper" / "figures" / "systematics"
FIGURES_ROOT.mkdir(parents=True, exist_ok=True)
pm = plots.PlotManager(root=FIGURES_ROOT, overwrite=True)

In [ ]:
STUDY = "magabs"

scale_cut = [0.3, 3]
version = "v_1p1"
name = "npz_bs_bp_mag"

tag = sy.scale_cut_tag(scale_cut)
DATA_DIR = sy.variant_dir(ROOT, STUDY, scale_cut, version)
SPL_DIR = sy.variant_dir(ROOT, STUDY, scale_cut, version, what="splines")

with open(DATA_DIR / f"{STUDY}_metadata_{tag}_{version}.json") as f:
    meta = json.load(f)
N_REALIZATIONS = meta["n_realizations"]
MAG_PERTURBATION = meta["mag_perturbation"]
ALPHA_LABEL = rf"$\alpha \rightarrow \alpha + \mathcal{{N}}(0,{MAG_PERTURBATION:g})$"

n_eval_points = 200
bin_colors = {1: "tab:blue", 2: "tab:orange", 3: "tab:red", 4: "tab:cyan"}
print(f"{STUDY}: alpha error {MAG_PERTURBATION}, {N_REALIZATIONS} realizations")
print(ALPHA_LABEL)

In [ ]:
# load every fit and summarize it on a grid common to all realizations of a bin
summaries, pooled, budget = {}, {}, []

for tomo in sy.TOMO_BINS:
    files = [SPL_DIR / f"spl_{name}_{tomo}_r{r:02d}" for r in range(N_REALIZATIONS + 1)]
    missing = [f for f in files if not Path(f"{f}.nc").exists()]
    if missing:
        raise FileNotFoundError(
            f"{len(missing)} spline(s) missing for bin {tomo}, e.g. {missing[0]}. "
            f"Run fit_splines.py --study {STUDY} first."
        )
    splines = [spline.BayesianBSpline.from_saved_model(str(f)) for f in files]

    z_eval = sy.common_grid(splines, n_points=n_eval_points)
    _, per_samples = sy.pool_realizations(splines, z_eval, n_eval_points=n_eval_points)

    per_summ = [sy.summarize_samples(s, z_eval) for s in per_samples]
    summaries[tomo] = per_summ

    # Pool the *perturbed* realizations only (index 0 is the unperturbed fit).
    pooled[tomo] = sy.summarize_samples(
        np.concatenate(per_samples[1:], axis=0), z_eval
    )

    row = sy.sigma_budget(per_summ[0], pooled[tomo], per_summ[1:])
    row["tomo_bin"] = tomo
    budget.append(row)

budget = pd.DataFrame(budget).set_index("tomo_bin")
budget

In [ ]:
tbl = pd.DataFrame(index=budget.index)
tbl["<z>"] = budget["mean_z_fiducial"]
tbl["sigma_stat"] = budget["sigma_fiducial"]
tbl["sigma_sys"] = budget["sigma_sys"]
tbl["sigma_tot"] = np.hypot(budget["sigma_fiducial"], budget["sigma_sys"])
tbl["increase_%"] = 100 * (tbl["sigma_tot"] / tbl["sigma_stat"] - 1)
tbl["shift_[sig]"] = (
    budget["mean_z_pooled"] - budget["mean_z_fiducial"]
) / budget["sigma_fiducial"]

In [ ]:
with pm.make_plot(
    f"{STUDY}_nz_{tag}_{version}",
    figsize=(9, 6), nrows=2, ncols=2, formats=["pdf", "png"], show=True,
) as (fig, axs):
    for tomo, ax in zip(sy.TOMO_BINS, axs.flat):
        fid, pool = summaries[tomo][0], pooled[tomo]
        z = fid["z_eval"]

        ax.fill_between(
            z, fid["lower"], fid["upper"],
            color=bin_colors[tomo], alpha=0.45, lw=0, zorder=1,
            label="statistical only",
        )
        ax.plot(z, fid["median"], color=bin_colors[tomo], lw=1.8, zorder=3)
        ax.plot(z, pool["lower"], color="k", lw=1.1, ls="--", zorder=2,
                label=r"marginalized over $\alpha$")
        ax.plot(z, pool["upper"], color="k", lw=1.1, ls="--", zorder=2)

        ax.axhline(0, color="k", lw=0.8, ls=":")
        ax.set_xlim(z.min(), z.max())
        ax.text(
            0.96, 0.92, f"Bin {tomo}", transform=ax.transAxes,
            ha="right", va="top", fontsize=12,
        )
        if tomo in (3, 4):
            ax.set_xlabel(r"$z$")
        if tomo in (1, 3):
            ax.set_ylabel(r"$n(z)$")
        if tomo == 1:
            ax.legend(frameon=False, fontsize=9, loc="upper left")
    fig.tight_layout()

In [ ]:
with pm.make_plot(
    f"{STUDY}_meanz_{tag}_{version}",
    figsize=(6.6, 4.2), formats=["pdf", "png"], show=True,
) as (fig, ax):
    x = np.arange(len(sy.TOMO_BINS))
    for i, tomo in enumerate(sy.TOMO_BINS):
        r = tbl.loc[tomo]
        ax.errorbar(x[i] - 0.11, 0.0, yerr=1.0,
                    fmt="o", ms=5, color="0.25", capsize=5, lw=1.5)
        ax.errorbar(x[i] + 0.11, r["shift_[sig]"], yerr=r["sigma_tot"] / r["sigma_stat"],
                    fmt="s", ms=5, color="tab:red", capsize=5, lw=1.5)

    ax.axhline(0, color="k", lw=0.8, ls=":")
    ax.set_xticks(
        x,
        [f"Bin {t}\n" + rf"$\sigma_{{\rm stat}}={budget.loc[t, 'sigma_fiducial']:.3f}$"
         for t in sy.TOMO_BINS],
    )
    ax.set_ylabel(
        r"$(\langle z \rangle - \langle z \rangle_{\rm stat}) / \sigma_{\rm stat}$"
    )
    ax.set_title(ALPHA_LABEL, fontsize=11)
    handles = [
        plt.Line2D([], [], color="0.25", marker="o", ls="", label="statistical only"),
        plt.Line2D([], [], color="tab:red", marker="s", ls="",
                   label=r"marginalized over $\alpha$"),
    ]
    ax.legend(handles=handles, frameon=False, fontsize=9, loc="lower right")
    ax.margins(x=0.12)
    ax.set_ylim(-1.9, 1.9)
    fig.tight_layout()

In [ ]:
with pm.make_plot(
    f"{STUDY}_meanz_posterior_{tag}_{version}",
    figsize=(9, 6), nrows=2, ncols=2, formats=["pdf", "png"], show=True,
) as (fig, axs):
    for tomo, ax in zip(sy.TOMO_BINS, axs.flat):
        fid_mz = summaries[tomo][0]["mean_z_samples"]
        pooled_mz = pooled[tomo]["mean_z_samples"]
        centres = np.array([s["mean_z"] for s in summaries[tomo][1:]])

        lo, hi = np.percentile(pooled_mz, [0.05, 99.95])
        edges = np.linspace(lo, hi, 70)
        ax.hist(fid_mz, bins=edges, density=True, color="0.55", alpha=0.55,
                label="unperturbed")
        ax.hist(pooled_mz, bins=edges, density=True, histtype="step",
                color=bin_colors[tomo], lw=1.8, label=r"pooled over $\alpha$")

        top = 1.32 * ax.get_ylim()[1]
        ax.set_xlim(lo, hi)
        ax.set_ylim(-0.05 * top, top)
        ax.plot(centres, np.full_like(centres, -0.026 * top), "|",
                color=bin_colors[tomo], ms=7, mew=1.0, alpha=0.8)

        ax.text(0.96, 0.92, f"Bin {tomo}", transform=ax.transAxes,
                ha="right", va="top", fontsize=12)
        ax.set_xlabel(r"$\langle z \rangle$")
        ax.set_ylabel("density")
        if tomo == sy.TOMO_BINS[0]:
            ax.legend(frameon=False, fontsize=9, loc="upper left")
    fig.tight_layout()

In [ ]:
out = {}
for tomo in sy.TOMO_BINS:
    fid, pool = summaries[tomo][0], pooled[tomo]
    out[f"{tomo}/z"] = fid["z_eval"]
    for label, s in (("fid", fid), ("pooled", pool)):
        for key in ("median", "mean", "lower", "upper", "std"):
            out[f"{tomo}/{label}_{key}"] = s[key]
    out[f"{tomo}/realization_mean_z"] = np.array(
        [s["mean_z"] for s in summaries[tomo]]
    )

np.savez_compressed(DATA_DIR / f"{STUDY}_summary_{tag}_{version}.npz", **out)
budget.to_csv(DATA_DIR / f"{STUDY}_sigma_budget_{tag}_{version}.csv")
tbl.to_csv(DATA_DIR / f"{STUDY}_table_{tag}_{version}.csv")